# Weak-Score Recommendation Agent
Analyzes diagnostic test scores and creates a personalized, actionable **7-Day Mastery Schedule** using **LangGraph** and **Llama 3.3 (HuggingFace)**.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
openai_key = os.getenv("OPENAI_API_KEY")
hf_token = os.getenv("HUGGINGFACEHUB_API_TOKEN") or os.getenv("HF_TOKEN")

if openai_key and not openai_key.startswith("your_"):
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.4)
    print("LLM Loaded (OpenAI):", llm.model_name)
elif hf_token and not hf_token.startswith("your_"):
    from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
    endpoint = HuggingFaceEndpoint(
        repo_id="meta-llama/Llama-3.3-70B-Instruct",
        task="conversational",
        huggingfacehub_api_token=hf_token
    )
    llm = ChatHuggingFace(llm=endpoint)
    print("LLM Loaded (HuggingFace):", endpoint.repo_id)
else:
    raise ValueError("Please configure HUGGINGFACEHUB_API_TOKEN or OPENAI_API_KEY in .env")


LLM Loaded (HuggingFace): meta-llama/Llama-3.3-70B-Instruct


In [2]:
from typing import TypedDict
from langgraph.graph import StateGraph, END

class RecommendationState(TypedDict):
    learner_profile: str
    weak_scores: str
    study_plan: str

def generate_plan(state: RecommendationState):
    prompt = f"""You are an adaptive curriculum and learning strategist.
A learner requires an actionable mastery plan based on this diagnostic:
Learner Goal / Domain: {state['learner_profile']}
Identified Weak Areas / Low Scores: {state['weak_scores']}

Create a structured 7-Day Targeted Mastery Plan in Markdown:
For each Day (Day 1 through Day 7):
- **Objective**: Core concept/skill
- **Concept Deep Dive**: What they need to understand
- **Hands-on Practice**: Specific exercise / problem
- **Time Commitment**: (e.g. 45 mins)
- **Success Benchmark**: How to confirm mastery

End with 2-3 curated, free resources (docs, cheat-sheets, platforms)."""
    res = llm.invoke(prompt)
    return {"study_plan": res.content if hasattr(res, 'content') else str(res)}

# Build and Compile LangGraph Workflow
graph = StateGraph(RecommendationState)
graph.add_node("plan_generator", generate_plan)
graph.set_entry_point("plan_generator")
graph.add_edge("plan_generator", END)
recommender_app = graph.compile()

# Execute Demo Plan
demo_input = {
    "learner_profile": "Backend Python Engineer",
    "weak_scores": "Database Indexing & Query Optimization (40/100), Docker Multi-stage Builds (35/100)"
}
result = recommender_app.invoke(demo_input)
print("=== 7-DAY PERSONALIZED STUDY PLAN ===\n")
print(result["study_plan"])


=== 7-DAY PERSONALIZED STUDY PLAN ===

### 7-Day Targeted Mastery Plan for Backend Python Engineer
#### Day 1: Introduction to Database Indexing
- **Objective**: Understand the basics of database indexing and its importance in query optimization.
- **Concept Deep Dive**: Learn what database indexes are, types of indexes (e.g., B-tree, hash), and how they improve query performance.
- **Hands-on Practice**: Create a simple database with Python using SQLite, and experiment with creating indexes on different columns.
- **Time Commitment**: 60 mins
- **Success Benchmark**: Successfully create an index on a column and measure the performance improvement in query execution time.

#### Day 2: Query Optimization Techniques
- **Objective**: Learn various techniques for optimizing database queries.
- **Concept Deep Dive**: Study how to use EXPLAIN and ANALYZE commands to understand query execution plans, and learn about indexing strategies.
- **Hands-on Practice**: Optimize a poorly performing qu